# AFM Simulation & 1D-CNN Regression Pipeline  v6
Generates synthetic AFM approach curves with physically-motivated parameters, trains a **1D convolutional neural network** to jointly predict **Young's modulus** and **contact-point offset** from normalised force curves using a **MAPE / MAE combined loss**, then produces spatial 2-D prediction maps.

**No geometry or force-volume blocks** — the workflow goes straight from physics simulation → CNN training → spatial prediction.

## 1  Imports

In [ ]:
import numpy as np
import scipy.ndimage as ndi
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import warnings, os, time
from dataclasses import dataclass, field

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__}  |  device: {DEVICE}")


## 2  User Parameters

In [ ]:
# ── A: Force-curve physics ────────────────────────────────────────────────────
CONTACT_MODEL      = "hertz_sphere"   # "hertz_sphere" | "sneddon_cone"
N_TIME             = 150              # contact-region sample points
N_PRECONTACT       = 50              # pre-contact sample points (F = 0)
PRECONTACT_NM      = 60.0             # z-range before contact (nm)
MAX_INDENT_NM      = 80.0             # maximum indentation (nm)

# ── B: Tip ────────────────────────────────────────────────────────────────────
TIP_SHAPE          = "sphere"         # "sphere" | "cone"
TIP_RADIUS_NM      = 20.0
TIP_HALF_ANGLE_DEG = 20.0

# ── C: Cantilever ─────────────────────────────────────────────────────────────
SPRING_CONST_N_M   = 0.1             # N/m
Q_FACTOR           = 1.0
RESONANT_FREQ_HZ   = 13e3

# ── D: Material ───────────────────────────────────────────────────────────────
NU_SAMPLE          = 0.5

# ── E: ML dataset ─────────────────────────────────────────────────────────────
N_CURVES           = 5000            # training + validation curves
E_MIN_KPA          = 0.5
E_MAX_KPA          = 500.0           # log-uniform sampling
CONTACT_OFFSET_STD_NM = 8.0          # std of z_cp draw (nm)
ENABLE_FORCE_NOISE = True
FORCE_NOISE_NN     = 0.3             # nN additive Gaussian noise

# ── F: CNN architecture ───────────────────────────────────────────────────────
CNN_CHANNELS       = [32, 64, 128, 256]   # conv layer output channels
CNN_KERNEL_SIZE    = 7
CNN_FC_HIDDEN      = 128
DROPOUT_RATE       = 0.3

# ── G: Training ───────────────────────────────────────────────────────────────
BATCH_SIZE         = 128
N_EPOCHS           = 80
LEARNING_RATE      = 3e-4
WEIGHT_DECAY       = 1e-4
VALIDATION_FRAC    = 0.15
TEST_FRAC          = 0.15
LOSS_WEIGHT_E      = 1.0             # weight on MAPE_E term
LOSS_WEIGHT_CP     = 0.5             # weight on MAE_cp term (different scale)
RANDOM_SEED        = 42

# ── H: 2-D prediction map ────────────────────────────────────────────────────
MAP_NX, MAP_NY     = 64, 64          # pixel grid
MAP_PIXEL_NM       = 8.0             # nm / pixel
E_SOFT_KPA         = 5.0             # Young's modulus inside soft inclusion
E_STIFF_KPA        = 150.0           # Young's modulus in background
INCLUSION_FRAC     = 0.28            # inclusion radius / half-grid width
CP_OFFSET_SOFT_NM  = 5.0            # contact-point offset in soft region
CP_OFFSET_STIFF_NM = -5.0           # contact-point offset in stiff region

# ── I: Output ─────────────────────────────────────────────────────────────────
SHOW_EXAMPLE_CURVES = True
SHOW_TRAINING_CURVE = True
SHOW_EVAL_PLOTS     = True
SHOW_PREDICTION_MAP = True
EXPORT              = True
EXPORT_PREFIX       = "afm_v6"

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
N_CURVE_PTS = N_PRECONTACT + N_TIME
print(f"Force-curve length: {N_CURVE_PTS} pts  ({N_PRECONTACT} pre + {N_TIME} contact)")
print(f"Dataset: {N_CURVES} curves  |  E=[{E_MIN_KPA},{E_MAX_KPA}] kPa  |  map: {MAP_NX}×{MAP_NY}")


## 3  Config Dataclasses

In [ ]:
@dataclass
class TipConfig:
    shape: str = TIP_SHAPE
    radius_nm: float = TIP_RADIUS_NM
    half_angle_deg: float = TIP_HALF_ANGLE_DEG

@dataclass
class CantileverConfig:
    k_N_m: float = SPRING_CONST_N_M
    Q: float = Q_FACTOR
    f0_Hz: float = RESONANT_FREQ_HZ

@dataclass
class SimConfig:
    contact_model: str = CONTACT_MODEL
    n_time: int = N_TIME
    n_precontact: int = N_PRECONTACT
    precontact_nm: float = PRECONTACT_NM
    max_indent_nm: float = MAX_INDENT_NM
    nu: float = NU_SAMPLE
    enable_noise: bool = ENABLE_FORCE_NOISE
    noise_nN: float = FORCE_NOISE_NN

tip_cfg  = TipConfig()
cant_cfg = CantileverConfig()
sim_cfg  = SimConfig()
print("Configs ready.")


## 4  Force-Curve Simulator

In [ ]:
def _e_star(E_kPa: float, nu: float) -> float:
    """Reduced modulus (Pa)."""
    return (E_kPa * 1e3) / (2.0 * (1.0 - nu**2))

def simulate_approach_curve(
        E_kPa: float,
        z_cp_nm: float,
        sim: SimConfig,
        tip: TipConfig,
        cant: CantileverConfig,
        rng: np.random.Generator) -> tuple[np.ndarray, np.ndarray]:
    """
    Return (z_piezo_nm, force_nN) for one approach curve.

    Layout
    ------
    Pre-contact : z_piezo linearly from (z_cp - precontact_nm) → z_cp,  F = 0
    Contact     : indentation δ from 0 → max_indent_nm
                  z_piezo = z_cp + δ + F_nN * 1e-9 / k_N_m * 1e9   (cantilever bending)
    """
    k = cant.k_N_m

    # ── pre-contact ──────────────────────────────────────────────────────────
    z_pre = np.linspace(z_cp_nm - sim.precontact_nm, z_cp_nm, sim.n_precontact)
    F_pre = np.zeros(sim.n_precontact)

    # ── contact ──────────────────────────────────────────────────────────────
    delta_nm = np.linspace(0.0, sim.max_indent_nm, sim.n_time)
    d_m      = delta_nm * 1e-9
    Es       = _e_star(E_kPa, sim.nu)

    if sim.contact_model == "hertz_sphere":
        R_m  = tip.radius_nm * 1e-9
        F_N  = (4.0 / 3.0) * Es * np.sqrt(R_m) * d_m**1.5
    else:  # sneddon_cone
        alpha = np.deg2rad(tip.half_angle_deg)
        F_N   = (2.0 / np.pi) * np.tan(alpha) * Es * d_m**2

    F_nN   = F_N * 1e9
    z_con  = z_cp_nm + delta_nm + (F_nN * 1e-9 / k) * 1e9   # scanner moves with bending

    z_all = np.concatenate([z_pre, z_con])
    F_all = np.concatenate([F_pre, F_nN])

    if sim.enable_noise:
        F_all = F_all + rng.normal(0.0, sim.noise_nN, F_all.shape)

    return z_all, F_all


def normalise_curve(z_nm: np.ndarray, F_nN: np.ndarray) -> np.ndarray:
    """
    Return 2 × N_CURVE_PTS array with channels:
      ch0: z_piezo normalised to [0, 1] by range
      ch1: force    normalised to [0, 1] by max positive value
    """
    z_n = (z_nm - z_nm.min()) / (z_nm.max() - z_nm.min() + 1e-12)
    f_max = np.maximum(F_nN, 0.0).max()
    f_n   = np.clip(F_nN, 0.0, None) / (f_max + 1e-12)
    return np.stack([z_n, f_n], axis=0).astype(np.float32)   # (2, N)

print("Simulator functions defined.")


## 5  Generate Training Dataset

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)

# Sample E log-uniformly and z_cp from Normal
log_E_min, log_E_max = np.log10(E_MIN_KPA), np.log10(E_MAX_KPA)
E_samples  = 10 ** rng.uniform(log_E_min, log_E_max, N_CURVES)
cp_samples = rng.normal(0.0, CONTACT_OFFSET_STD_NM, N_CURVES)

print(f"Generating {N_CURVES} force curves …", end=" ", flush=True)
t0 = time.time()
X_raw = np.zeros((N_CURVES, 2, N_CURVE_PTS), dtype=np.float32)

for i in range(N_CURVES):
    z, F = simulate_approach_curve(E_samples[i], cp_samples[i],
                                    sim_cfg, tip_cfg, cant_cfg, rng)
    X_raw[i] = normalise_curve(z, F)

print(f"done in {time.time()-t0:.1f}s")
print(f"X_raw shape: {X_raw.shape}   dtype: {X_raw.dtype}")
print(f"E range   : {E_samples.min():.2f}–{E_samples.max():.2f} kPa")
print(f"cp range  : {cp_samples.min():.1f}–{cp_samples.max():.1f} nm")

# Labels
y_logE = np.log10(E_samples).astype(np.float32)          # regression target: log10(E)
y_cp   = cp_samples.astype(np.float32)                   # regression target: z_cp (nm)

if SHOW_EXAMPLE_CURVES:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    idx8 = rng.choice(N_CURVES, 8, replace=False)
    norm_c = mcolors.LogNorm(vmin=E_MIN_KPA, vmax=E_MAX_KPA)
    cmap   = plt.cm.plasma

    # Raw z vs F (unnormalised) for a selection
    for i in idx8:
        z, F = simulate_approach_curve(E_samples[i], cp_samples[i],
                                        sim_cfg, tip_cfg, cant_cfg, rng)
        axes[0].plot(z, F, color=cmap(norm_c(E_samples[i])), lw=0.9, alpha=0.8)
    axes[0].set_xlabel("z-piezo (nm)"); axes[0].set_ylabel("Force (nN)")
    axes[0].set_title("Raw approach curves")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm_c)
    plt.colorbar(sm, ax=axes[0], label="E (kPa)")

    # Normalised ch0 (z) and ch1 (F)
    for i in idx8:
        axes[1].plot(X_raw[i, 0], color=cmap(norm_c(E_samples[i])), lw=0.9, alpha=0.8)
    axes[1].set_xlabel("sample index"); axes[1].set_ylabel("norm z")
    axes[1].set_title("Channel 0 – normalised z")

    for i in idx8:
        axes[2].plot(X_raw[i, 1], color=cmap(norm_c(E_samples[i])), lw=0.9, alpha=0.8)
    axes[2].set_xlabel("sample index"); axes[2].set_ylabel("norm F")
    axes[2].set_title("Channel 1 – normalised F")

    plt.tight_layout(); plt.show()


## 6  PyTorch Dataset & Splits

In [ ]:
class CurveDataset(Dataset):
    def __init__(self, X, y_logE, y_cp):
        self.X     = torch.from_numpy(X)
        self.y_logE = torch.from_numpy(y_logE).unsqueeze(1)
        self.y_cp   = torch.from_numpy(y_cp).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y_logE[idx], self.y_cp[idx]

full_ds = CurveDataset(X_raw, y_logE, y_cp)

n_total = len(full_ds)
n_test  = int(n_total * TEST_FRAC)
n_val   = int(n_total * VALIDATION_FRAC)
n_train = n_total - n_val - n_test

gen = torch.Generator().manual_seed(RANDOM_SEED)
train_ds, val_ds, test_ds = random_split(full_ds, [n_train, n_val, n_test], generator=gen)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Split — train: {n_train}  val: {n_val}  test: {n_test}")


## 7  1-D CNN Architecture
Two input channels: normalised z-piezo (ch0) and normalised force (ch1).  
Two output heads: **log₁₀(E/kPa)** and **z_contact (nm)**.  
Loss = `LOSS_WEIGHT_E × MAPE_E + LOSS_WEIGHT_CP × MAE_cp`

In [ ]:
class Conv1dBlock(nn.Module):
    """Conv → BN → ReLU → MaxPool."""
    def __init__(self, in_ch, out_ch, kernel, pool=2, dropout=0.0):
        super().__init__()
        pad = kernel // 2
        self.block = nn.Sequential(
            nn.Conv1d(in_ch, out_ch, kernel_size=kernel, padding=pad, bias=False),
            nn.BatchNorm1d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(pool),
            nn.Dropout(dropout) if dropout > 0 else nn.Identity(),
        )
    def forward(self, x):
        return self.block(x)


class ForceCurveCNN(nn.Module):
    """
    Input : (B, 2, N_CURVE_PTS)   — 2-channel normalised approach curve
    Output: (B, 1) log10_E,  (B, 1) z_cp_nm
    """
    def __init__(self, n_pts: int, channels: list, kernel: int,
                 fc_hidden: int, dropout: float):
        super().__init__()
        in_ch = 2
        blocks = []
        for out_ch in channels:
            blocks.append(Conv1dBlock(in_ch, out_ch, kernel, pool=2, dropout=dropout))
            in_ch = out_ch
        self.encoder = nn.Sequential(*blocks)
        self.pool    = nn.AdaptiveAvgPool1d(1)   # global average pool → (B, C, 1)

        self.head_E = nn.Sequential(
            nn.Linear(in_ch, fc_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(fc_hidden, 1)
        )
        self.head_cp = nn.Sequential(
            nn.Linear(in_ch, fc_hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(fc_hidden, 1)
        )

    def forward(self, x):
        z = self.pool(self.encoder(x)).squeeze(-1)   # (B, C)
        return self.head_E(z), self.head_cp(z)


def mape_loss(y_pred: torch.Tensor, y_true_logE: torch.Tensor) -> torch.Tensor:
    """
    MAPE in linear E space.
    y_pred and y_true are log10(E/kPa); we back-transform to compute percentage error.
    """
    E_pred = 10.0 ** y_pred
    E_true = 10.0 ** y_true_logE
    return torch.mean(torch.abs(E_pred - E_true) / (E_true + 1e-6)) * 100.0


def combined_loss(logE_pred, cp_pred, logE_true, cp_true,
                  w_E=LOSS_WEIGHT_E, w_cp=LOSS_WEIGHT_CP):
    loss_E  = mape_loss(logE_pred, logE_true)
    loss_cp = torch.mean(torch.abs(cp_pred - cp_true))     # MAE in nm
    return w_E * loss_E + w_cp * loss_cp, loss_E.item(), loss_cp.item()


model = ForceCurveCNN(
    n_pts=N_CURVE_PTS,
    channels=CNN_CHANNELS,
    kernel=CNN_KERNEL_SIZE,
    fc_hidden=CNN_FC_HIDDEN,
    dropout=DROPOUT_RATE,
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {n_params:,}")


## 8  Training

In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-6)

history = {"train_loss": [], "val_loss": [],
           "train_mape": [], "val_mape": [],
           "train_mae_cp": [], "val_mae_cp": []}

best_val_loss = float("inf")
best_state    = None

print(f"Training for {N_EPOCHS} epochs on {DEVICE} …")
t_start = time.time()

for epoch in range(1, N_EPOCHS + 1):
    # ── train ──
    model.train()
    tr_loss = tr_mape = tr_mae_cp = 0.0
    for X_b, yE_b, ycp_b in train_loader:
        X_b, yE_b, ycp_b = X_b.to(DEVICE), yE_b.to(DEVICE), ycp_b.to(DEVICE)
        optimizer.zero_grad()
        pE, pcp = model(X_b)
        loss, mape_e, mae_c = combined_loss(pE, pcp, yE_b, ycp_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tr_loss   += loss.item()
        tr_mape   += mape_e
        tr_mae_cp += mae_c
    scheduler.step()
    n_tr = len(train_loader)
    history["train_loss"].append(tr_loss / n_tr)
    history["train_mape"].append(tr_mape / n_tr)
    history["train_mae_cp"].append(tr_mae_cp / n_tr)

    # ── validate ──
    model.eval()
    vl_loss = vl_mape = vl_mae_cp = 0.0
    with torch.no_grad():
        for X_b, yE_b, ycp_b in val_loader:
            X_b, yE_b, ycp_b = X_b.to(DEVICE), yE_b.to(DEVICE), ycp_b.to(DEVICE)
            pE, pcp = model(X_b)
            loss, mape_e, mae_c = combined_loss(pE, pcp, yE_b, ycp_b)
            vl_loss   += loss.item()
            vl_mape   += mape_e
            vl_mae_cp += mae_c
    n_vl = len(val_loader)
    history["val_loss"].append(vl_loss / n_vl)
    history["val_mape"].append(vl_mape / n_vl)
    history["val_mae_cp"].append(vl_mae_cp / n_vl)

    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        best_state    = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if epoch % 10 == 0 or epoch == 1:
        elapsed = time.time() - t_start
        lr_now  = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch:3d}/{N_EPOCHS}  "
              f"train_loss={history['train_loss'][-1]:.3f}  "
              f"val_loss={history['val_loss'][-1]:.3f}  "
              f"MAPE_E={history['val_mape'][-1]:.1f}%  "
              f"MAE_cp={history['val_mae_cp'][-1]:.2f}nm  "
              f"lr={lr_now:.2e}  t={elapsed:.0f}s")

# Restore best weights
model.load_state_dict(best_state)
print(f"\nBest val loss: {best_val_loss:.4f}  (weights restored)")


## 9  Training Curves

In [ ]:
if SHOW_TRAINING_CURVE:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    epochs = range(1, N_EPOCHS + 1)

    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"],   label="val")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Combined loss")
    axes[0].set_title("Total loss  (w_E × MAPE + w_cp × MAE)")
    axes[0].legend()

    axes[1].plot(epochs, history["train_mape"], label="train")
    axes[1].plot(epochs, history["val_mape"],   label="val")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MAPE (%)")
    axes[1].set_title("MAPE — Young's modulus (linear kPa space)")
    axes[1].legend()

    axes[2].plot(epochs, history["train_mae_cp"], label="train")
    axes[2].plot(epochs, history["val_mae_cp"],   label="val")
    axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("MAE (nm)")
    axes[2].set_title("MAE — contact point")
    axes[2].legend()

    plt.tight_layout(); plt.show()


## 10  Test-Set Evaluation

In [ ]:
model.eval()
all_logE_true, all_logE_pred = [], []
all_cp_true,   all_cp_pred   = [], []

with torch.no_grad():
    for X_b, yE_b, ycp_b in test_loader:
        X_b = X_b.to(DEVICE)
        pE, pcp = model(X_b)
        all_logE_true.append(yE_b.numpy())
        all_logE_pred.append(pE.cpu().numpy())
        all_cp_true.append(ycp_b.numpy())
        all_cp_pred.append(pcp.cpu().numpy())

logE_true = np.concatenate(all_logE_true).ravel()
logE_pred = np.concatenate(all_logE_pred).ravel()
cp_true   = np.concatenate(all_cp_true).ravel()
cp_pred   = np.concatenate(all_cp_pred).ravel()

E_true_kPa = 10.**logE_true
E_pred_kPa = 10.**logE_pred

mape_test = np.mean(np.abs(E_pred_kPa - E_true_kPa) / (E_true_kPa + 1e-6)) * 100
mdre_test = np.median(np.abs(E_pred_kPa - E_true_kPa) / (E_true_kPa + 1e-6)) * 100
mae_cp    = np.mean(np.abs(cp_pred - cp_true))
r2_E      = 1 - np.sum((logE_true - logE_pred)**2) / (np.sum((logE_true - logE_true.mean())**2) + 1e-12)
r2_cp     = 1 - np.sum((cp_true  - cp_pred )**2)  / (np.sum((cp_true  - cp_true.mean())**2)  + 1e-12)

print("── Young's modulus ───────────────────────────────────────────")
print(f"  R² (log scale)  : {r2_E:.4f}")
print(f"  MAPE (kPa)      : {mape_test:.1f} %")
print(f"  MdRE (kPa)      : {mdre_test:.1f} %")
print("── Contact point ─────────────────────────────────────────────")
print(f"  R²              : {r2_cp:.4f}")
print(f"  MAE             : {mae_cp:.2f} nm")

if SHOW_EVAL_PLOTS:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # 1) E parity – log-log
    ax = axes[0]
    sc = ax.scatter(E_true_kPa, E_pred_kPa,
                    c=np.log10(E_true_kPa), cmap='plasma', s=8, alpha=0.5)
    lims = [E_MIN_KPA * 0.7, E_MAX_KPA * 1.3]
    ax.plot(lims, lims, 'k--', lw=1)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("E_true (kPa)"); ax.set_ylabel("E_pred (kPa)")
    ax.set_title(f"Young's modulus  R²={r2_E:.3f}  MAPE={mape_test:.1f}%")
    plt.colorbar(sc, ax=ax, label="log₁₀(E_true)")

    # 2) cp parity
    ax = axes[1]
    ax.scatter(cp_true, cp_pred, s=8, alpha=0.5, color='teal')
    lim_cp = max(abs(cp_true).max(), abs(cp_pred).max()) * 1.15
    ax.plot([-lim_cp, lim_cp], [-lim_cp, lim_cp], 'k--', lw=1)
    ax.set_xlabel("z_contact_true (nm)"); ax.set_ylabel("z_contact_pred (nm)")
    ax.set_title(f"Contact point  R²={r2_cp:.3f}  MAE={mae_cp:.2f} nm")

    # 3) E residuals vs E_true
    ax = axes[2]
    rel_err = (E_pred_kPa - E_true_kPa) / E_true_kPa * 100
    ax.scatter(E_true_kPa, rel_err, s=6, alpha=0.4, color='coral')
    ax.axhline(0, color='k', lw=1)
    ax.set_xscale('log')
    ax.set_xlabel("E_true (kPa)"); ax.set_ylabel("Relative error (%)")
    ax.set_title("E relative residuals")

    plt.tight_layout(); plt.show()


## 11  2-D Prediction Map
A parametric spatial E distribution (circular soft inclusion in a stiff background) is used to generate **one force curve per pixel**.  The trained CNN predicts **E** and **z_contact** for each curve, producing two prediction maps that can be compared against ground truth.

In [ ]:
# ── Build ground-truth spatial maps ──────────────────────────────────────────
Y_idx, X_idx = np.mgrid[0:MAP_NY, 0:MAP_NX]
cx, cy = MAP_NX / 2.0, MAP_NY / 2.0
r_norm = np.sqrt((X_idx - cx)**2 + (Y_idx - cy)**2) / (min(MAP_NX, MAP_NY) / 2.0)
in_inclusion = r_norm < INCLUSION_FRAC

E_true_map  = np.where(in_inclusion, E_SOFT_KPA, E_STIFF_KPA).astype(np.float32)
cp_true_map = np.where(in_inclusion, CP_OFFSET_SOFT_NM, CP_OFFSET_STIFF_NM).astype(np.float32)

# ── Generate one force curve per pixel and normalise ─────────────────────────
rng_map = np.random.default_rng(RANDOM_SEED + 999)
X_map = np.zeros((MAP_NY, MAP_NX, 2, N_CURVE_PTS), dtype=np.float32)

print(f"Generating {MAP_NY*MAP_NX} force curves for {MAP_NY}×{MAP_NX} pixel map …", end=" ")
t0 = time.time()
for iy in range(MAP_NY):
    for ix in range(MAP_NX):
        z, F = simulate_approach_curve(
            E_true_map[iy, ix], cp_true_map[iy, ix],
            sim_cfg, tip_cfg, cant_cfg, rng_map)
        X_map[iy, ix] = normalise_curve(z, F)
print(f"done in {time.time()-t0:.1f}s")

# ── Run CNN on the full map ───────────────────────────────────────────────────
X_map_flat = X_map.reshape(-1, 2, N_CURVE_PTS)   # (NY*NX, 2, N)
map_loader = DataLoader(
    torch.utils.data.TensorDataset(torch.from_numpy(X_map_flat)),
    batch_size=256, shuffle=False)

E_pred_flat  = []
cp_pred_flat = []

model.eval()
with torch.no_grad():
    for (batch,) in map_loader:
        pE, pcp = model(batch.to(DEVICE))
        E_pred_flat.append(pE.cpu().numpy())
        cp_pred_flat.append(pcp.cpu().numpy())

E_pred_map  = (10.0 ** np.concatenate(E_pred_flat ).ravel()).reshape(MAP_NY, MAP_NX)
cp_pred_map = np.concatenate(cp_pred_flat).ravel().reshape(MAP_NY, MAP_NX)

print(f"E_pred range : {E_pred_map.min():.1f}–{E_pred_map.max():.1f} kPa")
print(f"cp_pred range: {cp_pred_map.min():.2f}–{cp_pred_map.max():.2f} nm")

# ── Plot: ground truth vs prediction ─────────────────────────────────────────
if SHOW_PREDICTION_MAP:
    fig = plt.figure(figsize=(14, 11))
    gs  = GridSpec(2, 3, figure=fig, wspace=0.35, hspace=0.4)

    extent = [0, MAP_NX * MAP_PIXEL_NM, 0, MAP_NY * MAP_PIXEL_NM]

    # Row 0 – Young's modulus
    vE_min = min(E_true_map.min(), E_pred_map.min())
    vE_max = max(E_true_map.max(), E_pred_map.max())
    norm_E = mcolors.LogNorm(vmin=vE_min, vmax=vE_max)

    ax = fig.add_subplot(gs[0, 0])
    im = ax.imshow(E_true_map, norm=norm_E, cmap='viridis',
                   origin='lower', extent=extent)
    ax.set_title("E  — ground truth (kPa)"); ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
    plt.colorbar(im, ax=ax, label="E (kPa)")

    ax = fig.add_subplot(gs[0, 1])
    im = ax.imshow(E_pred_map, norm=norm_E, cmap='viridis',
                   origin='lower', extent=extent)
    ax.set_title("E  — CNN prediction (kPa)"); ax.set_xlabel("x (nm)")
    plt.colorbar(im, ax=ax, label="E (kPa)")

    ax = fig.add_subplot(gs[0, 2])
    E_rel_err = (E_pred_map - E_true_map) / E_true_map * 100
    vabs = np.percentile(np.abs(E_rel_err), 95)
    im = ax.imshow(E_rel_err, cmap='RdBu_r', origin='lower', extent=extent,
                   vmin=-vabs, vmax=vabs)
    ax.set_title("E  — relative error (%)"); ax.set_xlabel("x (nm)")
    plt.colorbar(im, ax=ax, label="Rel. error (%)")

    # Row 1 – Contact point
    vcp_abs = max(abs(cp_true_map).max(), abs(cp_pred_map).max()) * 1.05
    norm_cp = mcolors.Normalize(vmin=-vcp_abs, vmax=vcp_abs)

    ax = fig.add_subplot(gs[1, 0])
    im = ax.imshow(cp_true_map, norm=norm_cp, cmap='coolwarm',
                   origin='lower', extent=extent)
    ax.set_title("z_contact — ground truth (nm)"); ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
    plt.colorbar(im, ax=ax, label="z_cp (nm)")

    ax = fig.add_subplot(gs[1, 1])
    im = ax.imshow(cp_pred_map, norm=norm_cp, cmap='coolwarm',
                   origin='lower', extent=extent)
    ax.set_title("z_contact — CNN prediction (nm)"); ax.set_xlabel("x (nm)")
    plt.colorbar(im, ax=ax, label="z_cp (nm)")

    ax = fig.add_subplot(gs[1, 2])
    cp_abs_err = cp_pred_map - cp_true_map
    vabs_cp = np.percentile(np.abs(cp_abs_err), 95)
    im = ax.imshow(cp_abs_err, cmap='RdBu_r', origin='lower', extent=extent,
                   vmin=-vabs_cp, vmax=vabs_cp)
    ax.set_title("z_contact — absolute error (nm)"); ax.set_xlabel("x (nm)")
    plt.colorbar(im, ax=ax, label="Abs. error (nm)")

    plt.suptitle(f"2-D CNN Prediction Maps  ({MAP_NY}×{MAP_NX} pixels, 1 curve/px)",
                 fontsize=13, y=1.01)
    plt.show()


## 12  Export

In [ ]:
if EXPORT:
    px = EXPORT_PREFIX
    np.save(f"{px}_X_raw.npy",        X_raw)
    np.save(f"{px}_y_logE.npy",       y_logE)
    np.save(f"{px}_y_cp.npy",         y_cp)
    np.save(f"{px}_E_true_map.npy",   E_true_map)
    np.save(f"{px}_cp_true_map.npy",  cp_true_map)
    np.save(f"{px}_E_pred_map.npy",   E_pred_map)
    np.save(f"{px}_cp_pred_map.npy",  cp_pred_map)
    torch.save({"model_state": model.state_dict(),
                "history": history,
                "hyperparams": {
                    "n_pts": N_CURVE_PTS,
                    "channels": CNN_CHANNELS,
                    "kernel": CNN_KERNEL_SIZE,
                    "fc_hidden": CNN_FC_HIDDEN,
                    "dropout": DROPOUT_RATE,
                }}, f"{px}_model.pt")
    print("Exported:")
    for f in sorted(os.listdir('.')):
        if f.startswith(px):
            print(f"  {f}  ({os.path.getsize(f)//1024} kB)")
else:
    print("EXPORT = False  →  skipping.")
